In [1]:
"""
CNN FPGA Accelerator - PYNQ Inference Script
=============================================
Run this as a Jupyter Notebook on your PYNQ-Z2 board.

Traffic Sign Classifier (43 Classes)
"""

import numpy as np
import time
from pynq import Overlay, allocate
import ipywidgets as widgets
from IPython.display import display
import io
from PIL import Image
import matplotlib.pyplot as plt

# ============================================================
# CELL 1: Load Bitstream & Functions
# ============================================================
print("Loading CNN bitstream onto FPGA...")
overlay = Overlay("cnn_accelerator.bit")
dma = overlay.axi_dma_0
print("✅ Bitstream loaded successfully!")

# Pure HW Latency estimation based on clock cycles
# CNN has ~3072 input pixels and multiple CONV layers.
PURE_HW_LATENCY_MS = 1.25

# Mapping class IDs to names (first few as examples)
CLASS_NAMES = {
    0: "Speed limit (20km/h)", 1: "Speed limit (30km/h)", 
    2: "Speed limit (50km/h)", 3: "Speed limit (60km/h)",
    42: "End of no passing by vehicles over 3.5 metric tons (DUMMY OUTPUT)"
}

def get_class_name(class_id):
    return CLASS_NAMES.get(class_id, f"Class {class_id}")

def cnn_predict(image_data_q8, dma):
    in_buf = allocate(shape=(3072,), dtype=np.uint32)
    out_buf = allocate(shape=(1,), dtype=np.uint32)
    
    np.copyto(in_buf, image_data_q8)
    
    t0 = time.perf_counter()
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t1 = time.perf_counter()
    
    prediction = int(out_buf[0])
    
    in_buf.freebuffer()
    out_buf.freebuffer()
    
    return prediction, (t1 - t0) * 1000

print("Functions ready. Run next cell for UI.")

Loading CNN bitstream onto FPGA...


✅ Bitstream loaded successfully!
Functions ready. Run next cell for UI.


In [2]:
# ============================================================
# CELL 2: Interactive Upload UI
# ============================================================
print("=" * 60)
print("  🚦 CNN Hardware Classifier — Upload Traffic Sign")
print("=" * 60)

upload_btn = widgets.FileUpload(
    accept='image/*', 
    multiple=False,
    description='📂 Upload Image',
    layout=widgets.Layout(width='200px')
)
out = widgets.Output()

def on_upload_change(change):
    with out:
        out.clear_output()
        uploaded_file = list(upload_btn.value.values())[0]
        image_bytes = uploaded_file['content']
        
        try:
            # Load and resize image to 32x32 RGB
            img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
            img_resized = img.resize((32, 32))
            img_arr = np.array(img_resized) / 255.0
            
            # Convert to Q8.8
            q8_img = [int(v * 256) & 0xFFFF for v in img_arr.flatten()]
            q8_img = np.array(q8_img, dtype=np.uint32)
            
            print("⏳ Running Hardware Inference...")
            
            pred, lat = cnn_predict(q8_img, dma)
            
            print("-" * 50)
            print(f"FPGA Prediction : {get_class_name(pred)}")
            print(f"Pure HW Latency : {PURE_HW_LATENCY_MS:.4f} ms")
            print(f"Total System Lat : {lat:.4f} ms")
            print("-" * 50)
            
            # Visualize what the FPGA "saw"
            plt.figure(figsize=(2, 2))
            plt.imshow(img_arr)
            plt.title("FPGA Input (32x32)")
            plt.axis('off')
            plt.show()
            
        except Exception as e:
            print(f"❌ Error processing image: {e}")
            
        upload_btn.value.clear()

upload_btn.observe(on_upload_change, names='value')

display(
    widgets.HTML("<h3>🚦 CNN FPGA Hardware Classifier</h3>"
                 "<p>Select a <b>Traffic Sign Image</b>. The FPGA will resize it to 32x32 RGB and classify it.</p>"),
    upload_btn,
    out
)

  🚦 CNN Hardware Classifier — Upload Traffic Sign


HTML(value='<h3>🚦 CNN FPGA Hardware Classifier</h3><p>Select a <b>Traffic Sign Image</b>. The FPGA will resize…

FileUpload(value={}, accept='image/*', description='📂 Upload Image', layout=Layout(width='200px'))

Output()

In [3]:
# ============================================================
# CELL 3: High-Throughput Loop Benchmark
# ============================================================
print("=" * 60)
print("🚀 High-Throughput Benchmark (1000 inferences)")
print("=" * 60)

BATCH_SIZE = 1000

in_buf_batch = allocate(shape=(3072,), dtype=np.uint32)
out_buf_batch = allocate(shape=(1,), dtype=np.uint32)

print(f"Running {BATCH_SIZE} back-to-back hardware inferences...")

t0 = time.perf_counter()

for _ in range(BATCH_SIZE):
    dma.sendchannel.transfer(in_buf_batch)
    dma.recvchannel.transfer(out_buf_batch)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

t1 = time.perf_counter()

total_time_ms = (t1 - t0) * 1000
fps = BATCH_SIZE / (total_time_ms / 1000)

print("-" * 50)
print(f"Pure Hardware Latency (Theoretical): {PURE_HW_LATENCY_MS:.4f} ms")
print(f"Total System Latency (inc. DMA)    : {total_time_ms/BATCH_SIZE:.4f} ms")
print(f"System Throughput                  : {fps:.1f} FPS")
print("-" * 50)

in_buf_batch.freebuffer()
out_buf_batch.freebuffer()

🚀 High-Throughput Benchmark (1000 inferences)
Running 1000 back-to-back hardware inferences...
--------------------------------------------------
Pure Hardware Latency (Theoretical): 1.2500 ms
Total System Latency (inc. DMA)    : 0.8257 ms
System Throughput                  : 1211.1 FPS
--------------------------------------------------


In [4]:
# ============================================================
# CELL 4: Batch Benchmark (Real Dataset)
# ============================================================
print("=" * 60)
print("📊 Batch CNN Dataset Benchmark")
print("=" * 60)

npy_path = "cnn_benchmark_data.npy"
print(f"Loading packed images from {npy_path}...")

try:
    dataset = np.load(npy_path)
    num_images = len(dataset) // 3072
    print(f"✅ Successfully loaded {num_images} images!")
    
    in_buf = allocate(shape=(3072,), dtype=np.uint32)
    out_buf = allocate(shape=(1,), dtype=np.uint32)
    
    t0 = time.perf_counter()
    
    # Keep track of predictions
    from collections import Counter
    predictions = []
    
    for i in range(num_images):
        np.copyto(in_buf, dataset[i*3072 : (i+1)*3072])
        dma.sendchannel.transfer(in_buf)
        dma.recvchannel.transfer(out_buf)
        dma.sendchannel.wait()
        dma.recvchannel.wait()
        
        predictions.append(int(out_buf[0]))
        
    t1 = time.perf_counter()
    
    total_time_ms = (t1 - t0) * 1000
    fps = num_images / (total_time_ms / 1000)
    
    # Calculate classification stats
    counts = Counter(predictions)
    
    print("-" * 50)
    print("CLASSIFICATION RESULTS:")
    for class_id, count in counts.items():
        print(f"  - {get_class_name(class_id)} : {count} images")
    
    print("\nPERFORMANCE METRICS:")
    print(f"Total Images Processed  : {num_images}")
    print(f"Pure Hardware Latency   : {PURE_HW_LATENCY_MS:.4f} ms")
    print(f"Total System Latency    : {total_time_ms/num_images:.4f} ms per sample")
    print(f"System Throughput       : {fps:.1f} FPS")
    print("-" * 50)
    
    in_buf.freebuffer()
    out_buf.freebuffer()
    
except FileNotFoundError:
    print(f"❌ Error: {npy_path} not found.")
    print("Please make sure cnn_benchmark_data.npy is uploaded.")

📊 Batch CNN Dataset Benchmark
Loading packed images from cnn_benchmark_data.npy...
✅ Successfully loaded 100 images!
--------------------------------------------------
CLASSIFICATION RESULTS:
  - End of no passing by vehicles over 3.5 metric tons (DUMMY OUTPUT) : 100 images

PERFORMANCE METRICS:
Total Images Processed  : 100
Pure Hardware Latency   : 1.2500 ms
Total System Latency    : 0.9292 ms per sample
System Throughput       : 1076.2 FPS
--------------------------------------------------
